# Spequlator phase 2: training-split experiments (D-032)

**Pre-registered before running.** Everything here uses the CoDocBench **train** split only; `test.jsonl` is never copied or read.

- **Split:** v1.0's project-held-out split: 20 dev projects (seed 13), the rest for fitting.
- **Metrics:** dev pairwise accuracy and pooled AUC, 95% project-cluster bootstrap CIs (1,000 resamples).
- **A. Bi-encoder (MiniLM) sweep:** learning rate {2e-5, 5e-5} × epochs {1, 3} × max length {128, 256}, batch 8, hard negatives. Pick: best dev AUC; within 0.005, fewer epochs, then shorter length.
- **B. Ablations at the picked config:** no hard negative; batch 32; seeds 7 and 42.
- **C. Cross-encoder:** base {ms-marco-MiniLM-L6, all-MiniLM-L6} × input {changed lines + new code, old + new code}, 3 epochs, lr 2e-5, 512 tokens. Pick: best dev AUC; within 0.005, the shorter input. UniXcoder cross-encoder as a dev-only reference.
- **Outputs:** final models retrained on all train projects, cross-fitted dev scores for calibration, and a SHA-256 manifest.

Run with **Save Version → Save & Run All (Commit)**. Accelerator: GPU. Internet: on.


In [ ]:
# Cell 1: Setup: libraries, repo, data (pinned), GPU check
# Phase 2 (D-032): training-split-only experiments. The test split is never read here.
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # one GPU: keeps batch sizes exactly as configured
SMOKE = os.environ.get("PHASE2_SMOKE") == "1"  # tiny CPU dry run (used to test this notebook)
WORK = Path(os.environ.get("PHASE2_WORK", "/kaggle/working"))
REPO = WORK / "spequlator"
DATA = WORK / "codocbench"
OUT = WORK / "out"
PINNED = "ee0256da15a07dcf265d5313fe51a269aa7268e0"
TRAIN_SHA = "9eef860afedcbf059a4f2e408cafbaa6c61ffbf4369e687a9b45ae80f7ac7c6d"


def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if not SMOKE:
    sh("pip install -q sentence-transformers==6.2.0 transformers==5.20.2 docstring-parser")
    if not REPO.exists():
        sh(f"git clone -q https://github.com/shhuuubh-04/spequlator {REPO}")
    if not DATA.exists():
        sh(f"git clone -q https://github.com/kunpai/codocbench {WORK / 'cdb'}")
        sh(f"git -C {WORK / 'cdb'} checkout -q {PINNED}")
        DATA.mkdir(parents=True)
        sh(f"cp {WORK / 'cdb'}/dataset/train.jsonl {DATA}/")  # train only: test is never copied

got = hashlib.sha256((DATA / "train.jsonl").read_bytes()).hexdigest()
assert got == TRAIN_SHA, f"train.jsonl checksum mismatch: {got}"
os.environ["SPEQULATOR_DATA_DIR"] = str(DATA)
os.environ["SPEQULATOR_ARTIFACTS_DIR"] = str(OUT / "_scratch")
sys.path.insert(0, str(REPO / "src"))
OUT.mkdir(parents=True, exist_ok=True)

import numpy as np
import sentence_transformers
import torch
import transformers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VERSIONS = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
    "spequlator_commit": subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True
    ).stdout.strip(),
}
print(json.dumps(VERSIONS, indent=2))


In [ ]:
# Cell 2: Shared pieces: the same train split, project split and metrics as v1.0
from spequlator.datasets.codocbench import load_split
from spequlator.evaluation import stats
from spequlator.evaluation.finetune import project_split
from spequlator.evaluation.proxy_task import encoder_signals
from spequlator.models.encoder import EncoderSpec, SentenceEncoder
from spequlator.nlp.preprocessing import code_diff

MINILM = ("sentence-transformers/all-MiniLM-L6-v2", "1110a243fdf4706b3f48f1d95db1a4f5529b4d41")
SEED = 13
records = load_split("train")
fit, dev = project_split(records, 20, SEED)  # identical to v1.0's fine-tune gate split (D-026)
if SMOKE:
    fit, dev = fit[:40], dev[:30]
groups = np.array([r.project_key for r in dev])
print(f"train={len(records)} fit={len(fit)} dev={len(dev)} dev projects={len(set(groups))}")
RESULTS: list[dict] = []


def score(name: str, family: str, stale: np.ndarray, updated: np.ndarray, **cfg) -> dict:
    """Dev metrics with 95% project-cluster bootstrap CIs (same code as the frozen test run)."""
    n_boot = 50 if SMOKE else 1000
    row = {"name": name, "family": family, **cfg}
    for metric in ("pairwise_accuracy", "pooled_auc"):
        fn = getattr(stats, metric)
        point, lo, hi = stats.cluster_bootstrap(
            groups, lambda i, fn=fn: fn(stale[i], updated[i]), n_boot
        )
        row[metric], row[f"{metric}_ci"] = round(point, 4), [round(lo, 4), round(hi, 4)]
    RESULTS.append(row)
    (OUT / "dev_results.json").write_text(json.dumps(RESULTS, indent=2))
    print(f"{name:42} PA {row['pairwise_accuracy']:.3f}  AUC {row['pooled_auc']:.3f}")
    return row


In [ ]:
# Cell 3: Bi-encoder (MiniLM, as in v1.0): sweep and ablations on held-out projects
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers


def train_bi(recs, out_dir: Path, lr=2e-5, epochs=1, max_len=128, batch=8, hard_neg=True, seed=SEED):
    model = SentenceTransformer(MINILM[0], revision=MINILM[1], device=DEVICE)
    model.max_seq_length = max_len
    cols = {"anchor": [r.new_code for r in recs], "positive": [r.new_doc for r in recs]}
    if hard_neg:
        cols["negative"] = [r.old_doc for r in recs]
    args = SentenceTransformerTrainingArguments(
        output_dir=str(OUT / "_trainer"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        learning_rate=lr,
        warmup_steps=0.1,
        seed=seed,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        fp16=DEVICE == "cuda",
        use_cpu=DEVICE == "cpu",
    )
    SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=Dataset.from_dict(cols),
        loss=losses.MultipleNegativesRankingLoss(model),
    ).train()
    model.save(str(out_dir))
    return out_dir


def eval_bi(model_id: str, revision, max_len: int, label: str):
    enc = SentenceEncoder(EncoderSpec(label, model_id, revision, max_len), device=DEVICE, batch_size=64)
    df = encoder_signals(dev, enc)
    return df, df["stale_enc_delta"].to_numpy(), df["updated_enc_delta"].to_numpy()


BI_GRID = [
    {"lr": lr, "epochs": ep, "max_len": ml}
    for lr in (2e-5, 5e-5)
    for ep in (1, 3)
    for ml in (128, 256)
]
if SMOKE:
    BI_GRID = [{"lr": 2e-5, "epochs": 1, "max_len": 64}]

for ml in sorted({c["max_len"] for c in BI_GRID}):
    _, s, u = eval_bi(*MINILM, ml, f"pretrained-{ml}")
    score(f"bi pretrained len{ml}", "bi-encoder", s, u, max_len=ml, fine_tuned=False)

for cfg in BI_GRID:
    name = f"bi lr{cfg['lr']:g} ep{cfg['epochs']} len{cfg['max_len']}"
    t0 = time.time()
    path = train_bi(fit, OUT / "_bi_tmp", **cfg)
    _, s, u = eval_bi(str(path), None, cfg["max_len"], name)
    score(name, "bi-encoder", s, u, **cfg, batch=8, hard_neg=True, seconds=round(time.time() - t0))

# Pre-registered pick: best dev AUC; within 0.005 AUC prefer fewer epochs, then shorter length.
bi_rows = [r for r in RESULTS if r["family"] == "bi-encoder" and "lr" in r]
top = max(r["pooled_auc"] for r in bi_rows)
BEST_BI = min(
    (r for r in bi_rows if r["pooled_auc"] >= top - 0.005), key=lambda r: (r["epochs"], r["max_len"])
)
best = {k: BEST_BI[k] for k in ("lr", "epochs", "max_len")}
print("best bi-encoder config:", best)

ABLATIONS = [
    ("bi best, no hard negative", dict(best, hard_neg=False)),
    ("bi best, batch 32", dict(best, batch=32)),
]
for seed in (7, 42):
    ABLATIONS.append((f"bi best, seed {seed}", dict(best, seed=seed)))
if SMOKE:
    ABLATIONS = ABLATIONS[:1]
for name, cfg in ABLATIONS:
    path = train_bi(fit, OUT / "_bi_tmp", **cfg)
    _, s, u = eval_bi(str(path), None, cfg["max_len"], name)
    score(name, "bi-encoder-ablation", s, u, **cfg)


In [ ]:
# Cell 4: Cross-encoder: reads docstring + code change together, trained to classify
from huggingface_hub import HfApi
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def change_text(rec, fmt: str) -> str:
    """The code side of the pair. 'diff': changed lines first, then the new code."""
    if fmt == "full":
        return f"OLD CODE:\n{rec.old_code}\nNEW CODE:\n{rec.new_code}"
    d = code_diff(rec.old_code, rec.new_code)
    lines = [f"- {x}" for x in d.removed] + [f"+ {x}" for x in d.added]
    return "CHANGED LINES:\n" + "\n".join(lines) + f"\nNEW CODE:\n{rec.new_code}"


def pairs(recs, fmt: str):
    """Label 1 = the docstring fits the change (updated), 0 = stale (old docstring)."""
    out = []
    for r in recs:
        side = change_text(r, fmt)
        out += [(r.new_doc, side, 1.0), (r.old_doc, side, 0.0)]
    return out


def revision_of(model_id: str) -> str:
    return HfApi().model_info(model_id).sha


def train_ce(base: str, rev: str, recs, fmt: str, out_dir: Path, lr=2e-5, epochs=3, max_len=512, batch=16):
    torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(base, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(
        base, revision=rev, num_labels=1, ignore_mismatched_sizes=True
    ).to(DEVICE)
    data = pairs(recs, fmt)

    def collate(b):
        enc = tok([x[0] for x in b], [x[1] for x in b], truncation="longest_first",
                  max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([x[2] for x in b])
        return enc

    loader = DataLoader(data, batch_size=batch, shuffle=True, collate_fn=collate,
                        generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train()
    for ep in range(epochs):
        total = 0.0
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            labels = b.pop("labels")
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = lossf(model(**b).logits.squeeze(-1).float(), labels)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.item()
        print(f"  epoch {ep + 1}: mean loss {total / len(loader):.4f}")
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    meta = {"base": base, "revision": rev, "format": fmt, "lr": lr, "epochs": epochs,
            "max_len": max_len, "batch": batch, "label": "1 = docstring fits the change"}
    (out_dir / "spequlator_ce.json").write_text(json.dumps(meta, indent=2))
    return out_dir


@torch.no_grad()
def ce_scores(model_dir: Path, recs, fmt: str, max_len=512, batch=64) -> tuple[np.ndarray, np.ndarray]:
    """Logit per pair; higher = fits the new code (same direction as the Δ signals)."""
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()

    def run(texts):
        out = []
        for i in range(0, len(texts), batch):
            chunk = texts[i : i + batch]
            enc = tok([a for a, _ in chunk], [b for _, b in chunk], truncation="longest_first",
                      max_length=max_len, padding=True, return_tensors="pt").to(DEVICE)
            out.append(model(**enc).logits.squeeze(-1).float().cpu().numpy())
        return np.concatenate(out)

    sides = [change_text(r, fmt) for r in recs]
    return run([(r.old_doc, s) for r, s in zip(recs, sides)]), run([(r.new_doc, s) for r, s in zip(recs, sides)])


CE_BASES = {
    "ms-marco-MiniLM-L6": "cross-encoder/ms-marco-MiniLM-L6-v2",  # already a pair re-ranker
    "all-MiniLM-L6": MINILM[0],  # same base as the bi-encoder, new classification head
}
CE_GRID = [(b, f) for b in CE_BASES for f in ("diff", "full")]
CE_EPOCHS, CE_LEN = (1, 128) if SMOKE else (3, 512)
if SMOKE:
    CE_GRID = CE_GRID[:1]
CE_REV = {}
for base_key, fmt in CE_GRID:
    base = CE_BASES[base_key]
    rev = MINILM[1] if base == MINILM[0] else CE_REV.setdefault(base, revision_of(base))
    name = f"ce {base_key} {fmt}"
    t0 = time.time()
    path = train_ce(base, rev, fit, fmt, OUT / "_ce_tmp", epochs=CE_EPOCHS, max_len=CE_LEN)
    s, u = ce_scores(path, dev, fmt, max_len=CE_LEN)
    score(name, "cross-encoder", s, u, base=base, revision=rev, format=fmt, epochs=CE_EPOCHS,
          max_len=CE_LEN, seconds=round(time.time() - t0))

RUN_UNIXCODER = not SMOKE  # bigger code model: dev comparison only (too slow to serve on CPU)
if RUN_UNIXCODER:
    ux = ("microsoft/unixcoder-base", "5604afdc964f6c53782a6813140ade5216b99006")
    fmt = max((r for r in RESULTS if r["family"] == "cross-encoder"), key=lambda r: r["pooled_auc"])["format"]
    path = train_ce(*ux, fit, fmt, OUT / "_ce_ux", epochs=CE_EPOCHS, max_len=CE_LEN, batch=8)
    s, u = ce_scores(path, dev, fmt, max_len=CE_LEN, batch=16)
    score(f"ce unixcoder {fmt} (dev only)", "cross-encoder-reference", s, u, base=ux[0],
          revision=ux[1], format=fmt, epochs=CE_EPOCHS, max_len=CE_LEN)


In [ ]:
# Cell 5: Final models on all of train + cross-fitted calibration scores + manifest
import pandas as pd

ce_rows = [r for r in RESULTS if r["family"] == "cross-encoder"]
top = max(r["pooled_auc"] for r in ce_rows)
BEST_CE = min((r for r in ce_rows if r["pooled_auc"] >= top - 0.005),
              key=lambda r: (r["format"] != "diff", r["base"]))  # tie: shorter input, then name
print("best bi-encoder:", BEST_BI["name"], "| best cross-encoder:", BEST_CE["name"])

# Cross-fitted calibration: dev projects scored by models that never saw them (as in v1.0).
dev_bi = train_bi(fit, OUT / "_bi_dev", **best)
cal_bi, _, _ = eval_bi(str(dev_bi), None, best["max_len"], "minilm-ft2")
cal_bi["encoder"] = "minilm-ft2"
cal_bi.to_parquet(OUT / "signals_train_minilm-ft2.parquet", index=False)
dev_ce = train_ce(BEST_CE["base"], BEST_CE["revision"], fit, BEST_CE["format"], OUT / "_ce_dev",
                  epochs=CE_EPOCHS, max_len=CE_LEN)
s, u = ce_scores(dev_ce, dev, BEST_CE["format"], max_len=CE_LEN)
pd.DataFrame({"record_id": [r.record_id for r in dev], "stale_ce": s, "updated_ce": u}).to_parquet(
    OUT / "signals_train_ce.parquet", index=False
)

# Final models: same settings, all train projects.
final_bi = train_bi(records if not SMOKE else fit, OUT / "minilm-ft2", **best)
final_ce = train_ce(BEST_CE["base"], BEST_CE["revision"], records if not SMOKE else fit,
                    BEST_CE["format"], OUT / "ce", epochs=CE_EPOCHS, max_len=CE_LEN)


def sha_dir(path: Path) -> str:
    h = hashlib.sha256()
    for f in sorted(p for p in path.rglob("*") if p.is_file()):
        h.update(str(f.relative_to(path)).encode())
        h.update(f.read_bytes())
    return h.hexdigest()


manifest = {
    "phase": "2",
    "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "smoke": SMOKE,
    "versions": VERSIONS,
    "train_sha256": TRAIN_SHA,
    "split": {"fit_records": len(fit), "dev_records": len(dev), "dev_projects": 20, "seed": SEED},
    "best_bi": BEST_BI,
    "best_ce": BEST_CE,
    "sha256": {
        "minilm-ft2": sha_dir(final_bi),
        "ce": sha_dir(final_ce),
        "signals_train_minilm-ft2.parquet": hashlib.sha256((OUT / "signals_train_minilm-ft2.parquet").read_bytes()).hexdigest(),
        "signals_train_ce.parquet": hashlib.sha256((OUT / "signals_train_ce.parquet").read_bytes()).hexdigest(),
    },
}
(OUT / "phase2_manifest.json").write_text(json.dumps(manifest, indent=2))
table = pd.DataFrame(RESULTS).drop(columns=["pairwise_accuracy_ci", "pooled_auc_ci"], errors="ignore")
table.to_csv(OUT / "dev_results.csv", index=False)
print(table.to_string(index=False))


In [ ]:
# Cell 6: Package for download (Output tab → phase2_outputs.zip)
import shutil

for tmp in ("_scratch", "_trainer", "_bi_tmp", "_ce_tmp", "_ce_ux", "_bi_dev", "_ce_dev"):
    shutil.rmtree(OUT / tmp, ignore_errors=True)
shutil.make_archive(str(WORK / "phase2_outputs"), "zip", OUT)
print("ready:", WORK / "phase2_outputs.zip", round((WORK / "phase2_outputs.zip").stat().st_size / 1e6), "MB")
